# 06 — The Capstone on Real Hardware, and the Frontier

In `04/15` you proved something delicate and real: in *simulation*, a richer multi-frequency quantum embedding's coupling measure separates two matched-PLV ensembles by about **0.004 nats** of mutual information — a structure that the phase-locking value, by construction, cannot see. That result stands. Here we ask the honest hardware question that closes the course: can a real quantum processor even *see* an effect that small? Not "does the effect exist" — it does, you measured it — but "is today's hardware sharp enough to resolve it?" We will answer with numbers, and the answer will teach us exactly where the frontier lies.

**Prerequisites:** `04/15_discriminating_experiment`, `04/16_capstone_synthesis`, `04/17_vqe_limitation_de_palma`, `05/04_quantum_mutual_information_on_real_hardware`.

**What you'll be able to do**
- Measure `I(A:B)` on the device with a genuine statistical error bar (bootstrap-resampled from the counts).
- Quantify the device's *systematic* bias — the gap between the ideal QMI and what the chip returns.
- Compare both of those errors, side by side, to the 0.004-nat capstone effect.
- Estimate the resources — shots, and error mitigation — a real on-hardware demonstration would actually require.

You have measured `I(A:B)` on a chip already, in `05/04`, and watched decoherence steal a piece of it. Now you turn that same measurement into an instrument and point it at the capstone: you ask how finely a real device can resolve a mutual-information difference, and you compare that resolution to the size of the effect you are chasing. This is the most candid notebook in the course — and candour, here, is the science.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2
from qot_course.hardware.runtime import select_backend
from qot_course.hardware.tomography import two_qubit_tomography_circuits, density_from_counts
from qot_course.infotheory.quantum import quantum_mutual_information
from qot_course.quantum.composite import bell_state
from qot_course.quantum.density import density_matrix

np.random.seed(42)
viz.use_course_style()

CAPSTONE_EFFECT_NATS = 0.004  # from 04/15, in simulation (7.6 sigma vs the a2-matched null)

## 1 · What we are testing

Let us be precise about the claim we are *not* relitigating. In `04/15`, working with **exact density matrices** and a Monte-Carlo sample of **≈1.2M phase samples in the discrimination test** (150k phases per channel in `04/15`), the richer multi-frequency embedding's quantum mutual information separated two ensembles that were matched on their phase-locking value. The separation was about **0.004 nats**, and it was significant at **7.6σ against an a2-matched null**. That is a statement about a *simulated* experiment, and it holds. PLV is blind to that structure by construction; the quantum coupling measure is not. The capstone result is real.

This notebook asks a **different** question. Not "is the effect there?" but "**what is a real QPU's resolution for `I(A:B)`?**" — how finely can a physical device, with its finite shots and its gate and read-out errors, pin down a mutual-information value at all? These are two distinct quantities, and conflating them is the one mistake that would undo the lesson:

- the **7.6σ** of `04/15` is a *significance* — the simulated effect divided by the spread of a matched null, a statement about the simulated experiment;
- the **σ_stat** we are about to measure is a *device error bar* — the statistical scatter of a single hardware QMI estimate, a statement about the chip.

They are different σ's, living in different sentences. Keep them apart, and the rest of the notebook reads clean.

## 2 · Measuring the device's QMI resolution

To measure the device's resolution we use the cleanest probe we have: a Bell pair, whose ideal mutual information we know exactly. We reconstruct its joint state by two-qubit tomography — the nine Pauli settings of `05/04` — and read off `I(A:B)`. We work in **nats** throughout (natural log), so the device's number is directly comparable to the 0.004-nat capstone effect, which was reported in nats.

A single QMI estimate is not enough; we need to know how much it would *wobble*. So we characterise two separate errors:

- a **statistical band** `σ_stat`, obtained by bootstrap-resampling each setting's counts (drawing new multinomial counts with the same totals) and recomputing QMI many times — the scatter of those re-estimates is the statistical uncertainty, and it costs no new jobs on the device;
- a **systematic bias**, the gap `I_ideal − I_device`: the device's QMI sits *below* the ideal because decoherence and read-out error leak correlation, and that offset does not average away.

The first shrinks with more shots; the second does not. Holding them apart is the whole point.

In [ ]:
USE_HARDWARE = False  # flip to True after you've set up credentials (see 05/01)
backend, label, is_real = select_backend(use_hardware=USE_HARDWARE)
print(f"Running on: {label}  (real hardware = {is_real})")
if not is_real:
    backend.set_options(seed_simulator=7)  # reproducible offline figure; a real QPU has no such option

prep = QuantumCircuit(2)
prep.h(0)
prep.cx(0, 1)                               # a Bell pair: our QMI probe
circuits = two_qubit_tomography_circuits(prep)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
settings = list(circuits)
res = SamplerV2(mode=backend).run([pm.run(circuits[s]) for s in settings], shots=8192).result()
counts_by_setting = {s: res[i].data.c.get_counts() for i, s in enumerate(settings)}

qmi_ideal = quantum_mutual_information(density_matrix(bell_state()), dims=[2, 2], base=np.e)   # nats
qmi_point = quantum_mutual_information(density_from_counts(counts_by_setting, n_qubits=2), dims=[2, 2], base=np.e)

# Statistical band: bootstrap-resample each setting's counts (multinomial) -- cheap, no new jobs.
rng = np.random.default_rng(0)
boot = []
for _ in range(200):
    resampled = {}
    for s, counts in counts_by_setting.items():
        ks = list(counts)
        ps = np.array([counts[k] for k in ks], dtype=float)
        n = int(ps.sum())
        resampled[s] = {k: int(v) for k, v in zip(ks, rng.multinomial(n, ps / ps.sum()))}
    boot.append(quantum_mutual_information(density_from_counts(resampled, n_qubits=2), dims=[2, 2], base=np.e))
sigma_stat = float(np.std(boot))
bias = float(qmi_ideal - qmi_point)
print(f"I(A:B) ideal  = {qmi_ideal:.4f} nats")
print(f"I(A:B) device = {qmi_point:.4f} +/- {sigma_stat:.4f} (stat) nats")
print(f"systematic bias (ideal - device) = {bias:.4f} nats")
print(f"   discrimination band ~ sqrt(2)*sigma_stat = {np.sqrt(2) * sigma_stat:.4f} nats  (the systematic bias largely cancels in the ensemble DIFFERENCE)")
print(f"capstone effect to resolve       = {CAPSTONE_EFFECT_NATS} nats")
print(f"   effect / sigma_stat = {CAPSTONE_EFFECT_NATS / sigma_stat:.2f}    effect / bias = {CAPSTONE_EFFECT_NATS / bias:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
names = ["capstone\neffect", "device\nstatistical σ", "device\nsystematic bias"]
mags = [CAPSTONE_EFFECT_NATS, sigma_stat, bias]
ax.bar(names, mags, color=[COLORS["highlight"], COLORS["quantum"], COLORS["negative"]], alpha=0.9)
ax.set_yscale("log")
ax.set_ylabel("nats (log scale)")
ax.set_title("What the device must resolve (0.004 nats) vs the errors it actually has", pad=12)
for i, m in enumerate(mags):
    ax.text(i, m * 1.15, f"{m:.4f}", ha="center")
plt.show()

**Read the figure.** The 0.004-nat capstone effect is the smallest bar by a wide margin. It sits *below* the device's statistical scatter `σ_stat` — so a single hardware estimate cannot even pin the value to within the effect's own size — and it lies roughly two orders of magnitude below the systematic bias. On a log scale the three magnitudes are visibly stratified: effect, then statistical band, then bias, each well separated from the next. The separation the capstone relies on is plainly invisible on this hardware. And note *why* it is invisible: this is a statement about the device's **QMI resolution**, not about which states we chose to prepare — it is *independent of the embedding*. Swapping in the richer multi-frequency states would not move these bars, because the wall is the chip's noise floor, not the physics we are encoding. (Offline these are the modeled FakeManila errors; flip `USE_HARDWARE=True` and the bars report the literal chip's resolution.)

One caveat keeps the bias honest: the capstone effect is a *difference* of two matched-PLV QMIs, and a systematic offset that hits both ensembles alike largely cancels in that difference. So the 0.39-nat bias is the wall for reading an *absolute* QMI, while the true limit for the *discrimination* is the statistical band of the difference, about sqrt(2)\*sigma_stat (~0.012 nats here). The 0.004-nat effect sits below even that — so it stays unresolvable, but the systematic bias is not the binding constraint for telling the two ensembles apart.

## 3 · What would it take?

So the effect is below the noise. The natural next question is quantitative: *how much* would we have to improve to resolve it? Take the statistical part first. To resolve a 0.004-nat gap at about 3σ we need the statistical band itself to be no larger than a third of the effect, `σ_stat ≤ 0.004 / 3` nats. Tomographic statistical error scales as `σ_stat ∝ 1/√(shots)`, so to shrink the band by a factor we must increase the shots by that factor *squared*. The required shot count therefore scales as `(σ_stat / target)²`.

But — and this is the load-bearing caveat — **more shots only shrink the statistical band**. The *systematic* bias is a different animal: it comes from decoherence and read-out error, and it does not budge no matter how many times you sample. Driving it down needs error mitigation, or materially lower gate and read-out error rates — better hardware, not more of the same.

In [ ]:
target = CAPSTONE_EFFECT_NATS / 3.0          # aim: resolve the effect at ~3 sigma
shots_used = 8192
shots_needed = shots_used * (sigma_stat / target) ** 2
print(f"target sigma_stat         : {target:.5f} nats")
print(f"shots/setting now         : {shots_used:,}")
print(f"shots/setting needed (~)  : {shots_needed:,.0f}")
print(f"systematic bias remaining : {bias:.4f} nats  (>> {CAPSTONE_EFFECT_NATS}; unaffected by shots)")
print("=> even with the shots, the bias must be removed by error mitigation / better hardware.")

**Read the numbers.** Spell it out plainly. The statistical part alone calls for on the order of $10^5$–$10^6$ shots *per measurement setting* — a large but conceivable budget. Yet even after you spend it, the systematic bias remains: it is roughly **100× the size of the effect** you are trying to resolve, and it will sit there untouched until error mitigation or genuinely lower error rates remove it. This is the same structural wall that `04/17` (De Palma, Marvian, Rouzé, Stilck França) describes from the optimisation side: there, the obstacle to the variational program was the noise of the device; here, it is the noise floor of the measurement. In both cases the barrier is not a shortage of effort or cleverness — it is the physics of today's NISQ machines.

## 4 · The honest bottom line

Here is the whole story in two sentences that must never collapse into one.

The capstone result is **real and stands**: in simulation (`04/15`), a richer quantum embedding's coupling measure sees structure that the phase-locking value cannot — a 0.004-nat separation, significant at 7.6σ against a matched null. And **today's NISQ hardware cannot yet measure an effect that small** — the effect is below the device's statistical scatter (the binding constraint for the discrimination) and far below its systematic bias — and you can now say *exactly* why, and by *how much* it falls short. Note: the systematic bias largely cancels in the ensemble *difference*, so the statistical band (~sqrt(2)\*sigma_stat) is the true limit for telling the two ensembles apart; the bias is the wall for an absolute QMI reading.

That candour is not a disappointment; it is the science. A demonstrated effect, and an honest, quantitative map of the gap between that effect and the machines we have. Knowing precisely where the frontier sits — what resolution, what bias, what shot budget — is what makes it a frontier you can plan to cross, rather than a vague horizon.

## Your turn (optional, heavy — not part of the offline run)

This one is a real expedition, flagged so you know it is *not* part of the lightweight offline run above. If you want to push past the Bell-pair probe and attempt the capstone itself on a device:

- Encode the richer multi-frequency qutrit of `04/15` into **2 qubits per oscillator**, giving a 4-qubit joint state for the pair.
- Attempt the two-ensemble separation on the **simulator first** — confirm you can still see the 0.004-nat effect with exact (or high-shot) statistics before spending anything on hardware.
- Then use the **σ_stat you measured here** to estimate the QPU shots the on-hardware version would take, and the error mitigation the bias would demand.

Be warned: this is a large job — four qubits, full tomography, many settings, very high shot counts — well beyond a casual free-tier run. Treat it as a design exercise in *what a real demonstration would cost*, which is itself the lesson.

## The journey, end to end

Look back at the whole arc. You began in `00` with a single qubit — one amplitude, one Bloch vector — and a promise that information has a geometry. In `01` you built quantum states, composite systems, and entanglement. In `02` you turned entropy into geometry: Fisher–Rao, von Neumann entropy, mutual information, the Bures distance. In `03` you learned classical optimal transport — couplings, the Wasserstein distance, Sinkhorn — and watched it shake hands with information geometry. In `04` you lifted all of it into **quantum** optimal transport, reconciled its three faces, and ran the discriminating experiment that *demonstrated*, in simulation, a quantum coupling measure seeing what PLV cannot. And in `05` you met real hardware: tomography, fidelity, mutual information, the Bures distance — each measured on a live chip — culminating, here, in an honest map of the frontier.

That is the arc: from one qubit, through the geometry of information and the mathematics of transport, to a demonstrated quantum effect and a clear-eyed account of the gap between it and the machines of today. You did the hard version every step — you built the objects, validated them against closed forms, and refused to round a small effect up into a false victory or down into a non-result. Thank you for walking the whole road. The frontier is now yours to push.

## References

- G. De Palma, M. Marvian, C. Rouzé & D. Stilck França, "Limitations of variational quantum algorithms: a quantum optimal transport approach", *PRX Quantum* **4**, 010309 (2023). DOI:10.1103/PRXQuantum.4.010309 — arXiv:2204.03455. (Four authors — the frontier result of `17`.)
- J. A. Smolin, J. M. Gambetta & G. Smith (2012), "Efficient Method for Computing the Maximum-Likelihood Quantum State from Measurements with Additive Gaussian Noise," *Phys. Rev. Lett.* **108**, 070502, doi:10.1103/PhysRevLett.108.070502.

**Previous:** `notebooks/05_RealQuantumHardware/05_geometry_on_real_hardware_bures.ipynb`. *Course end.*